In [7]:
import pandas as pd
import numpy as np

# Load the dataset
original_df = pd.read_csv('C:\\Users\\esomw\\OneDrive\\Documents\\DataScience_Workspace\\Fraud_Detection_Project\\data\\nova_pay_combined.csv')

# Create a copy of the dataset
df = original_df.copy()

# Display basic dataset information
print("Original dataset shape:", original_df.shape)
print("Working copy shape:", df.shape)

# Display the first few rows of the dataset
df.head()

Original dataset shape: (11400, 26)
Working copy shape: (11400, 26)


,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [9]:
# 1. Drop irrelevant columns (Identfiers)

# Identifiers columns do not usually contribute to the predictive power of the model and can be safely removed.
identifier_columns = [
    "transaction_id",
    "customer_id",
    "device_id",
    "ip_address"
]

# Drop the identifier columns
df.drop(columns=identifier_columns, inplace=True)

print("Dataset shape after dropping identifier columns:", df.shape)
df.head()





Dataset shape after dropping identifier columns: (11400, 22)


,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,new_device,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,1.351351,False,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,12.758621,True,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,7.142857,False,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,0.925926,False,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,83.333333,False,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [15]:
# 2.  Check and deal with duplicates

# check numbers of duplicates
duplicates_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates_count}")

# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Reset index after removing duplicates
df.reset_index(drop=True, inplace=True)

print("Dataset shape after removing duplicates:", df.shape)



Number of duplicate rows: 0
Dataset shape after removing duplicates: (11200, 22)


In [16]:
# 3. Deal with Null Values

# Check missing values
print("Missing values before cleaning:")
print(df.isnull().sum())


Missing values before cleaning:
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   300
fee                          290
exchange_rate_src_to_dest      0
new_device                     0
ip_country                   296
location_mismatch              0
ip_risk_score                  0
kyc_tier                     295
account_age_days               0
device_trust_score           290
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64


In [18]:
# Separate numerical and categorical columns
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = df.select_dtypes(include=["object", "bool"]).columns

# Fill missing numerical values with median 
#Median is a robust measure of central tendency that is less affected by outliers compared to mean, making it a good choice for filling missing values in numerical data.
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with mode
#Mode is the most frequently occurring value in a dataset, making it a suitable choice for filling missing values in categorical data.
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Confirm missing values have been handled
print("Missing values after cleaning:")
print(df.isnull().sum())


Missing values after cleaning:
timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
new_device                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
dtype: int64


C:\Users\esomw\AppData\Local\Temp\ipykernel_37448\931984824.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object", "bool"]).columns


In [20]:
# 4. Fix Inconsistencies in Categorical Variables

# Identify categorical columns
categorical_columns = df.select_dtypes(include=["object"]).columns

# Standardize text formatting in categorical columns
for col in categorical_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

# Display unique values in categorical columns to check for inconsistencies
for col in categorical_columns:
    print(f"\nUnique values in {col}:")
    print(df[col].unique())


Unique values in timestamp:
<StringArray>
['2022-10-03_18:40:59.468549+00:00', '2022-10-03_20:39:38.468549+00:00',
 '2022-10-03_23:02:43.468549+00:00', '2022-10-04_01:08:53.468549+00:00',
 '2022-10-04_09:35:03.468549+00:00', '2022-10-04_12:09:59.468549+00:00',
 '2022-10-04_12:37:41.468549+00:00', '2022-10-04_16:27:44.468549+00:00',
 '2022-10-04_21:00:36.468549+00:00', '2022-10-04_21:20:42.468549+00:00',
 ...
 '2025-11-24_13:42:17.573611+00:00', '2025-11-25_00:45:10.573611+00:00',
 '2025-11-25_07:44:29.573611+00:00', '2025-11-25_07:56:54.573611+00:00',
 '2025-11-25_09:48:28.573611+00:00', '2025-11-25_10:05:35.573611+00:00',
 '2025-11-26_07:09:56.573611+00:00', '2025-11-27_06:19:11.573611+00:00',
 '2025-11-28_00:53:28.573611+00:00', '2025-11-29_20:10:47.573611+00:00']
Length: 11141, dtype: str

Unique values in home_country:
<StringArray>
['us', 'ca', 'uk', 'unknown']
Length: 4, dtype: str

Unique values in source_currency:
<StringArray>
['usd', 'cad', 'gbp']
Length: 3, dtype: str

Uniq

C:\Users\esomw\AppData\Local\Temp\ipykernel_37448\650083192.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object"]).columns


In [21]:
# Correct spelling inconsistencies in kyc_tier

df["kyc_tier"] = df["kyc_tier"].replace({
    "standrd": "standard",
    "enhancd": "enhanced",
    "nan": "unknown"
})

# Verify cleaned values
print("Unique values in kyc_tier after correction:")
print(df["kyc_tier"].unique())

Unique values in kyc_tier after correction:
<StringArray>
['standard', 'enhanced', 'low', 'unknown']
Length: 4, dtype: str


In [27]:
# ==============================
# Final Check
# ==============================

# Display cleaned dataset information
print("Cleaned dataset shape:", df.shape)
print("\nDataset information:")
df.info()

# Preview cleaned dataset
df.head(20)

Cleaned dataset shape: (11200, 22)

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 11200 entries, 0 to 11199
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp                  11200 non-null  str    
 1   home_country               11200 non-null  str    
 2   source_currency            11200 non-null  str    
 3   dest_currency              11200 non-null  str    
 4   channel                    11200 non-null  str    
 5   amount_src                 11200 non-null  str    
 6   amount_usd                 11200 non-null  float64
 7   fee                        11200 non-null  float64
 8   exchange_rate_src_to_dest  11200 non-null  float64
 9   new_device                 11200 non-null  bool   
 10  ip_country                 11200 non-null  str    
 11  location_mismatch          11200 non-null  bool   
 12  ip_risk_score              11200 non-null  float64
 13  

,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,new_device,ip_country,location_mismatch,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,2022-10-03_18:40:59.468549+00:00,us,usd,cad,atm,278.19,278.190,4.25,1.351351,False,us,False,0.123,standard,263,0.5220,0,0.223,0,0,0.00,0
1,2022-10-03_20:39:38.468549+00:00,ca,cad,mxn,web,208.51,154.290,4.24,12.758621,True,ca,False,0.569,standard,947,0.4750,0,0.268,0,1,0.00,0
2,2022-10-03_23:02:43.468549+00:00,us,usd,cny,mobile,160.33,160.330,2.70,7.142857,False,us,False,0.437,enhanced,367,0.9390,0,0.176,0,0,0.00,0
3,2022-10-04_01:08:53.468549+00:00,us,usd,eur,mobile,59.41,59.410,2.22,0.925926,False,us,False,0.594,standard,147,0.5510,0,0.391,0,0,0.00,0
4,2022-10-04_09:35:03.468549+00:00,us,usd,inr,mobile,200.96,200.960,3.61,83.333333,False,us,False,0.121,enhanced,257,0.8940,0,0.257,0,0,0.00,0
5,2022-10-04_12:09:59.468549+00:00,us,usd,gbp,mobile,526.9,526.900,8.75,0.800000,False,us,False,0.094,low,616,0.7020,0,0.361,0,0,0.00,0
6,2022-10-04_12:37:41.468549+00:00,ca,cad,gbp,mobile,149.24,110.440,2.33,0.592000,False,ca,False,0.299,standard,947,0.6250,0,0.268,0,0,0.00,0
7,2022-10-04_16:27:44.468549+00:00,ca,cad,inr,web,276.51,204.620,5.59,61.666667,False,ca,False,0.087,standard,1041,0.8670,0,0.245,0,1,0.12,0
8,2022-10-04_21:00:36.468549+00:00,us,usd,eur,mobile,99.52,99.520,2.41,0.925926,False,us,False,0.182,enhanced,367,0.9390,0,0.176,0,0,0.00,0
9,2022-10-04_21:20:42.468549+00:00,us,usd,php,web,302.55,302.550,5.26,58.823529,False,us,False,0.413,standard,1016,0.9440,0,0.247,0,0,0.10,1
